# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Zoye-J/FlyRank--MachineLearning/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
# Connecting to warehouse + build the modeling frame

from google.colab import userdata
import os
import pandas as pd
import numpy as np
import duckdb

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute("CREATE SECRET hf (TYPE huggingface, PROVIDER credential_chain);")

FACT_MAR = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet"
FACT_APR = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-04/*.parquet"
FACT_MAY = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-05/*.parquet"
DIM_CONTENT = "hf://datasets/FlyRank/internship-warehouse/dim_content.parquet"

# Same frame as Week 4 + client_hash_id for grouped split
MODEL_Q = f"""
WITH perf AS (
  SELECT
    content_hash_id,
    ANY_VALUE(client_hash_id)         AS client_hash_id,
    SUM(gsc_impressions)              AS impressions_30d,
    SUM(gsc_clicks)                   AS clicks_30d,
    AVG(NULLIF(gsc_avg_position, 0))  AS avg_position_30d
  FROM '{FACT_MAR}'
  GROUP BY content_hash_id
),
future AS (
  SELECT content_hash_id, SUM(gsc_impressions) AS imp_apr_may
  FROM (
    SELECT content_hash_id, gsc_impressions FROM '{FACT_APR}'
    UNION ALL
    SELECT content_hash_id, gsc_impressions FROM '{FACT_MAY}'
  )
  GROUP BY content_hash_id
)
SELECT
  p.content_hash_id,
  p.client_hash_id,
  p.impressions_30d,
  p.clicks_30d,
  p.avg_position_30d,
  d.content_type,
  d.main_intent,
  d.word_count,
  DATE_DIFF('day', d.content_updated_date, DATE '2026-03-31') AS days_since_update,
  f.imp_apr_may
FROM perf p
LEFT JOIN '{DIM_CONTENT}' d ON p.content_hash_id = d.content_hash_id
LEFT JOIN future f ON p.content_hash_id = f.content_hash_id
WHERE p.impressions_30d IS NOT NULL AND p.impressions_30d > 0
"""

df = con.execute(MODEL_Q).df()
print(f"Base rows: {len(df):,}")

# Feature engineering (Week 4))
df["ctr_30d"] = np.where(df["impressions_30d"] > 0,
                         100.0 * df["clicks_30d"] / df["impressions_30d"],
                         np.nan)

df["position_tier"] = pd.cut(
    df["avg_position_30d"],
    bins=[0, 3, 10, 20, 50, 1e9],
    labels=["top_3", "page_1", "page_2", "page_3_5", "deep"],
)

tier_medians = df.groupby("position_tier", observed=True)["ctr_30d"].median()
df["tier_median_ctr"] = df["position_tier"].map(tier_medians)

# Low-ctr flag (same as baseline)
df["low_ctr"] = (df["ctr_30d"] < df["tier_median_ctr"]).astype(int)
df["visible"] = (df["impressions_30d"] >= 500).astype(int)

# Baseline score (recreate for the comparison table)
df["baseline_score"] = (df["visible"] * df["low_ctr"] * df["impressions_30d"]).fillna(0)

# Label
df["is_declining_future"] = np.where(
    df["imp_apr_may"].isna() | (df["impressions_30d"] == 0),
    np.nan,
    (df["imp_apr_may"] < 0.8 * df["impressions_30d"]).astype(float),
)

# Keeping rows with the label
df = df.dropna(subset=["is_declining_future"]).copy()
print(f"Rows with label: {len(df):,}")
print(f"Base rate: {df['is_declining_future'].mean():.3f}")
print(f"Unique clients: {df['client_hash_id'].nunique()}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Base rows: 176,738
Rows with label: 176,738
Base rate: 0.283
Unique clients: 47


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*


Primary model: Logistic Regression.

My lane is Search Intent, framed as 'ranking/scoring' with a binary
classification underneath. My Week-4 baseline is:

score = visible × low_ctr × impressions_30d



That is a hand-tuned AND-rule over three roughly linear signals. Logistic
Regression is the natural comparison:

- Same signal space: LR sees the same features (visibility, CTR-vs-tier,
  impressions, position, staleness) and weights them instead of AND-ing
  them. If the signals are real, LR should beat the rule.
- Interpretable: Coefficients show which signals the model leans on. This
  is the Week 2 lesson, the model is a rule you can read.
- Not rewarding complexity: A shallow model that clearly beats the
  baseline is better than a deep model that barely does. The assignment says so.

Secondary model: Decision Tree (depth 3):

Non-linear, but bounded and readable. If LR underperforms, the tree shows
whether an interaction the rule missed is doing the work.

What I will NOT use:

- Gradient Boosting: The assignment says 'where safe', with this label, boosting can fit the threshold
  itself rather than the underlying signal. Not worth the interpretability cost.
- Deep models: No justification for the added complexity.


In [3]:
# printing the method choice as a reference record

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

method_plan = {
    "primary":   "Logistic Regression (interpretable, linear weights over the same signals)",
    "secondary": "Decision Tree, max_depth=3 (non-linear, still readable)",
    "rejected":  ["Gradient Boosting (overkill, label-threshold overfit risk)",
                  "Random Forest (only if LR underperforms badly)"],
    "why":       "Baseline is a linear AND-rule; LR is the apples-to-apples comparison.",
}

for k, v in method_plan.items():
    print(f"{k}: {v}")

primary: Logistic Regression (interpretable, linear weights over the same signals)
secondary: Decision Tree, max_depth=3 (non-linear, still readable)
rejected: ['Gradient Boosting (overkill, label-threshold overfit risk)', 'Random Forest (only if LR underperforms badly)']
why: Baseline is a linear AND-rule; LR is the apples-to-apples comparison.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*


Grouped by: client_hash_id

Pages from the same client share content strategy, editorial voice, and
audience. If a client appears in both train and test, the model can memorize
client-specific patterns and inflate its score. Grouping forces the model
to generalize to unseen clients which is what it must do in production.

Why not temporal:

The features always come from March 2026, and the label always comes from
April–May 2026. Those windows are already fixed and honest. What varies
between train and test is which clients, that is the correct axis of
generalization for this problem.

Split parameters:

- 80 / 20 train-test split
- Grouped by 'client_hash_id' via 'GroupShuffleSplit'
- Random seed 42 for reproducibility

Same split for the baseline:

The baseline is evaluated on the same test set that the model uses. This is
the same data, same metric, same split requirement.

In [4]:
# building the grouped split

from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=df["client_hash_id"]))

train_df = df.iloc[train_idx].copy()
test_df  = df.iloc[test_idx].copy()

print(f"Train rows:  {len(train_df):,}  ({len(train_df)/len(df):.1%})")
print(f"Test rows:   {len(test_df):,}  ({len(test_df)/len(df):.1%})")
print(f"Train clients: {train_df['client_hash_id'].nunique()}")
print(f"Test clients:  {test_df['client_hash_id'].nunique()}")
print()

# Confirm no client overlap — this is the whole point
overlap = set(train_df["client_hash_id"]) & set(test_df["client_hash_id"])
print(f"Client overlap between train and test: {len(overlap)} clients")
assert len(overlap) == 0, "GROUP LEAK: a client is in both sets"
print("No client overlap, grouped split is honest.")

print()
print(f"Train base rate: {train_df['is_declining_future'].mean():.3f}")
print(f"Test base rate:  {test_df['is_declining_future'].mean():.3f}")

Train rows:  138,310  (78.3%)
Test rows:   38,428  (21.7%)
Train clients: 37
Test clients:  10

Client overlap between train and test: 0 clients
No client overlap, grouped split is honest.

Train base rate: 0.310
Test base rate:  0.186


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

Answer:
Both baseline and model are evaluated on the **same test set**, with the same metric: Precision@K (K = 20, 50, 100).

The base rate is printed next to each metric from the building-baselines
skill: "precision@50 of 0.60 means little until you know whether random
picking gives 0.55 or 0.10."

In [5]:
# train LR + DT, compare against baseline

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score


# Same signals the baseline leaned on and more
FEATURES = [
    "impressions_30d",
    "clicks_30d",
    "ctr_30d",
    "avg_position_30d",
    "days_since_update",
]

# Fill NA (for GSC numerics, NULL ≠ 0. Use -1 as a sentinel)
def make_X(d):
    X = d[FEATURES].copy()
    X["avg_position_30d"] = X["avg_position_30d"].fillna(-1)     # -1 = "no position data"
    X["days_since_update"] = X["days_since_update"].fillna(-1)
    X["ctr_30d"] = X["ctr_30d"].fillna(0)                        # no clicks = 0% CTR
    return X

X_train = make_X(train_df)
X_test  = make_X(test_df)

y_train = train_df["is_declining_future"].values
y_test  = test_df["is_declining_future"].values

# Precision@K helper
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return float(np.nanmean(topk))

# Baselines
# Base rate = random picking
base_rate = float(np.mean(y_test))

# Week 4 hand rule
baseline_test_scores = test_df["baseline_score"].values

# Dummy (majority class)
from sklearn.dummy import DummyClassifier
dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_train, y_train)
dummy_scores = dummy.predict_proba(X_test)[:, 1]

# Logistic Regression
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

lr = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)
lr.fit(X_train_s, y_train)
lr_scores = lr.predict_proba(X_test_s)[:, 1]

# Decision Tree (depth 3)
dt = DecisionTreeClassifier(max_depth=3, class_weight="balanced", random_state=42)
dt.fit(X_train, y_train)
dt_scores = dt.predict_proba(X_test)[:, 1]

#The comparison table
def row(name, scores):
    return {
        "model": name,
        "P@20":  round(precision_at_k(scores, y_test, 20), 4),
        "P@50":  round(precision_at_k(scores, y_test, 50), 4),
        "P@100": round(precision_at_k(scores, y_test, 100), 4),
        "AUC":   round(roc_auc_score(y_test, scores), 4),
    }

table = pd.DataFrame([
    row("Base rate (random)", np.full(len(y_test), base_rate)),
    row("Dummy (majority)",   dummy_scores),
    row("Week-4 baseline",    baseline_test_scores),
    row("Logistic Regression", lr_scores),
    row("Decision Tree (d=3)", dt_scores),
])

print()
print("MODEL vs BASELINE: same test set, same metric")
print()
print(f"Test set: {len(y_test):,} rows, base rate = {base_rate:.3f}")
print()
display(table)

# Lift over baseline
lift_20 = table.loc[3, "P@20"] / table.loc[2, "P@20"]
lift_50 = table.loc[3, "P@50"] / table.loc[2, "P@50"]
lift_100 = table.loc[3, "P@100"] / table.loc[2, "P@100"]
print(f"\nLR lift over baseline:  P@20 ×{lift_20:.2f}  P@50 ×{lift_50:.2f}  P@100 ×{lift_100:.2f}")


MODEL vs BASELINE: same test set, same metric

Test set: 38,428 rows, base rate = 0.186



,model,P@20,P@50,P@100,AUC
0,Base rate (random),0.30,0.40,0.42,0.5000
1,Dummy (majority),0.30,0.40,0.42,0.5000
2,Week-4 baseline,0.45,0.52,0.45,0.5088
3,Logistic Regression,0.25,0.26,0.36,0.5605
4,Decision Tree (d=3),0.20,0.26,0.19,0.5611



LR lift over baseline:  P@20 ×0.56  P@50 ×0.50  P@100 ×0.80


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*


A short error analysis beats a big metric table. Three questions:

1. Where is the model wrong? (which kinds of pages it misranks)
2. What does it lean on? (feature importance / coefficients)
3. What does that tell me about the lane? (does the signal hold up?)

In [9]:
# coefficients, tree, and error buckets


print("LOGISTIC REGRESSION COEFFICIENTS (standardized)")
coefs = pd.DataFrame({
    "feature": FEATURES,
    "coefficient": lr.coef_[0],
    "abs_coef": np.abs(lr.coef_[0]),
}).sort_values("abs_coef", ascending=False)
display(coefs)

print()
print("DECISION TREE (depth 3) — readable rule")
print()
from sklearn.tree import export_text
print(export_text(dt, feature_names=FEATURES))

LOGISTIC REGRESSION COEFFICIENTS (standardized)


,feature,coefficient,abs_coef
1,clicks_30d,-0.933795,0.933795
3,avg_position_30d,-0.217137,0.217137
4,days_since_update,0.054293,0.054293
0,impressions_30d,0.033540,0.033540
2,ctr_30d,-0.003527,0.003527



DECISION TREE (depth 3) — readable rule

|--- clicks_30d <= 3.50
|   |--- avg_position_30d <= 12.74
|   |   |--- impressions_30d <= 1.50
|   |   |   |--- class: 0.0
|   |   |--- impressions_30d >  1.50
|   |   |   |--- class: 1.0
|   |--- avg_position_30d >  12.74
|   |   |--- days_since_update <= -77.50
|   |   |   |--- class: 1.0
|   |   |--- days_since_update >  -77.50
|   |   |   |--- class: 0.0
|--- clicks_30d >  3.50
|   |--- days_since_update <= -93.50
|   |   |--- ctr_30d <= 0.35
|   |   |   |--- class: 1.0
|   |   |--- ctr_30d >  0.35
|   |   |   |--- class: 0.0
|   |--- days_since_update >  -93.50
|   |   |--- clicks_30d <= 9.50
|   |   |   |--- class: 0.0
|   |   |--- clicks_30d >  9.50
|   |   |   |--- class: 0.0



In [10]:
# error buckets: where does the model do worst?

test_df = test_df.copy()
test_df["lr_score"] = lr_scores

# Rank by LR score; the top 50 is what a human would act on
test_df["lr_rank"] = test_df["lr_score"].rank(ascending=False, method="first")

def bucket(r):
    if r["lr_rank"] <= 50:
        return "top_50"
    if r["lr_rank"] <= 200:
        return "51_200"
    if r["lr_rank"] <= 1000:
        return "201_1000"
    return "rest"

test_df["rank_bucket"] = test_df.apply(bucket, axis=1)

err = test_df.groupby("rank_bucket").agg(
    n=("is_declining_future", "size"),
    actual_decline_rate=("is_declining_future", "mean"),
    median_impressions=("impressions_30d", "median"),
    median_ctr=("ctr_30d", "median"),
    median_position=("avg_position_30d", "median"),
).round(3)

print()
print("ERROR BUCKETS: how the model does across the ranking")
print()
display(err)

# False positives in the top 50
top50 = test_df[test_df["lr_rank"] <= 50]
fp = top50[top50["is_declining_future"] == 0]
print(f"\nTop-50 false positives: {len(fp)} / 50")
if len(fp) > 0:
    print("Their traits:")
    display(fp[["impressions_30d","ctr_30d","avg_position_30d","days_since_update"]].describe().round(2))


ERROR BUCKETS: how the model does across the ranking



,n,actual_decline_rate,median_impressions,median_ctr,median_position
rank_bucket,,,,,
201_1000,800,0.339,403.5,0.0,3.701
51_200,150,0.473,349.0,0.0,1.677
rest,37428,0.182,332.0,0.0,8.714
top_50,50,0.260,3.0,0.0,6.260



Top-50 false positives: 37 / 50
Their traits:


,impressions_30d,ctr_30d,avg_position_30d,days_since_update
count,37.00,37.0,27.00,37.00
mean,5659.38,0.0,6.10,198.05
std,24808.84,0.0,4.70,89.60
min,1.00,0.0,0.38,-85.00
25%,2.00,0.0,2.50,235.00
50%,3.00,0.0,6.40,235.00
75%,6.00,0.0,8.00,243.00
max,134984.00,0.0,19.00,243.00


My read

- Top-50 precision: 0.260: the model's top picks decline at only 26.0%,
  which is below the test-set base rate of 0.283 and well below the Week-4
  baseline's 0.560 at P@50. The model did not beat the baseline at the
  top of the queue. This is a real finding, not noise.

- Where it's wrong: the top-50 false positives (37 of 50) are dominated by
  pages with median 3 impressions, essentially no traffic. The model has
  learned "no clicks → likely decline," which is trivially true but useless as
  a review queue. A useful ranking would need a minimum-impressions floor that
  the LR does not enforce.

- The middle of the ranking is where the model works: The 51–200 bucket
  has a 47.3% decline rate — clearly above the base rate. The signal exists,
  but the model's *ranking* places noise above it.

- What it leans on: the LR coefficients are dominated by 'clicks_30d'
  (standardized coefficient −0.934), with 'avg_position_30d' a distant second
  (−0.217). 'ctr_30d' the feature my Week4 rule was built around is
  essentially zero (−0.004). The model did **not** find the Search-Intent
  signal the lane hypothesized; it found a low-traffic proxy instead.

- **The decision tree agrees.** It splits first on `clicks_30d ≤ 3.50`, then
  falls back on position and `impressions_30d ≤ 1.50`. No branch keys on CTR.

- What that tells me about the lane:  the Search-Intent hypothesis that
  visible low-CTR pages are the right review target is not supported by
  what a linear model finds on this slice. The baseline's hand rule
  (visibility AND low CTR AND impressions) actually captured the useful
  signal better than the LR did, because it forced a visibility floor.
  A future model should either include that floor as a feature or train on a
  restricted subset (impressions_30d ≥ 500).

- Honest verdict: the baseline wins on the metric that matters (P@50).
  The model's advantage is confined to a middle band that a review team would
  not act on first. I keep the baseline frozen and note this for Week 6.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.